In [ ]:
import os, gc, time, json, hashlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model, Input, mixed_precision
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, auc)

for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError: pass
mixed_precision.set_global_policy('mixed_float16')

# ------------------------- CONFIG -------------------------
DATASET_PATH = "path/to/dataset"   # NIH malaria cell images: folder with one sub-folder per class (Uninfected/, Parasitized/)
OUT_DIR      = "outputs"           # results, weights and figures are written here
os.makedirs(OUT_DIR, exist_ok=True)
IMG_SIZE     = (160, 160)
RANDOM_STATE = 42
QUICK_TEST   = False            # True -> tiny run to check that the code works
CLASSES      = ['Uninfected', 'Parasitized']     # index 1 = positive (malaria)
TUNE_FRACTION, K_FOLDS = 0.15, 5

# Hyperparameters are NOT fixed: they are found by the automatic search below and stored in this dict.
HP = {}

# ---- memory-saving settings ----
MICRO_BATCH   = 8      # batch 32 = 4 micro-batches of 8 (try 4 for an even larger saving)
TARGET_SAVING = 60.0   # % target used for the pass/fail marks
LOSS_SCALE    = 128.0  # static loss scale for float16 training

tf.keras.utils.set_random_seed(RANDOM_STATE)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus, '| TF', tf.__version__)
assert gpus or QUICK_TEST, 'No GPU found. A GPU is required (memory is measured on the GPU); use QUICK_TEST = True only to smoke-test the code.'

In [ ]:
# ---------------- data: two DISJOINT pools (tuning pool for the hyperparameter search, CV pool for the final evaluation) ----------------
records = []
for cls in CLASSES:
    d = os.path.join(DATASET_PATH, cls)
    for f in os.listdir(d):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            records.append({'filepath': os.path.join(d, f), 'label': cls})
df = pd.DataFrame(records)
print(df['label'].value_counts())

df_cv, df_tune = train_test_split(df, test_size=TUNE_FRACTION, stratify=df['label'], random_state=RANDOM_STATE)
df_cv, df_tune = df_cv.reset_index(drop=True), df_tune.reset_index(drop=True)
if QUICK_TEST:
    df_cv = df_cv.sample(min(len(df_cv), 600), random_state=0).reset_index(drop=True)
print("CV pool:", len(df_cv), " tuning pool (hyperparameter search only):", len(df_tune))
folds = list(StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(df_cv, df_cv['label']))

def make_gens(train_df, val_df, batch_size):
    tr = ImageDataGenerator(rescale=1./255, horizontal_flip=True, zoom_range=0.2)
    va = ImageDataGenerator(rescale=1./255)
    kw = dict(x_col='filepath', y_col='label', target_size=IMG_SIZE, class_mode='binary',
              classes=CLASSES, batch_size=batch_size)
    return (tr.flow_from_dataframe(train_df, shuffle=True, seed=RANDOM_STATE, **kw),
            va.flow_from_dataframe(val_df, shuffle=False, **kw))

In [ ]:
# ---------------------------- model components ----------------------------
class HaarLevel(layers.Layer):
    """Level-`level` 2-D Haar (average/difference, as in the original code): 12 channels (LL,LH,HL,HH x RGB)."""
    def __init__(self, level, **kw):
        super().__init__(**kw); self.level = level
    def call(self, x):
        cur = x
        for _ in range(self.level):
            a, b = cur[:, :, ::2, :], cur[:, :, 1::2, :]
            L, H = (a + b) / 2, (a - b) / 2
            LL = (L[:, ::2] + L[:, 1::2]) / 2; LH = (L[:, ::2] - L[:, 1::2]) / 2
            HL = (H[:, ::2] + H[:, 1::2]) / 2; HH = (H[:, ::2] - H[:, 1::2]) / 2
            out = tf.concat([LL, LH, HL, HH], axis=-1); cur = LL
        return out
    def get_config(self): return {**super().get_config(), 'level': self.level}

class ChannelAvgMax(layers.Layer):
    def call(self, x):
        return tf.concat([tf.reduce_mean(x, -1, keepdims=True), tf.reduce_max(x, -1, keepdims=True)], -1)

def spatial_attention(x, k):
    att = layers.Conv2D(1, k, padding='same', activation='sigmoid', use_bias=False)(ChannelAvgMax()(x))
    return layers.Multiply()([x, att])

def cbr(x, f, stride=1, sep=False):
    Conv = layers.SeparableConv2D if sep else layers.Conv2D
    x = Conv(f, 3, strides=stride, padding='same')(x)
    return layers.Activation('relu')(layers.BatchNormalization()(x))

def make_seg(name, in_shapes, fn):
    ins = [Input(shape=s) for s in in_shapes]
    return Model(ins, fn(*ins), name=name)

class Remat(layers.Layer):
    """Runs a sub-model; if `ckpt`, its activations are NOT kept for the backward pass but recomputed
    (tf.recompute_grad). BatchNorm moving averages are not updated a second time during recomputation."""
    def __init__(self, sub, ckpt=False, **kw):
        super().__init__(**kw); self.sub = sub; self.ckpt = ckpt; self.force = False
        self._bns = [l for l in sub.layers if isinstance(l, layers.BatchNormalization)]
    def call(self, inputs, training=None):
        multi = isinstance(inputs, (list, tuple)); args = list(inputs) if multi else [inputs]
        if not (self.ckpt and (training is True or self.force)):
            return self.sub(args if multi else args[0], training=training)
        state = {'n': 0}
        def run(*a):
            recompute = state['n'] > 0; state['n'] += 1
            old = [b.momentum for b in self._bns]
            if recompute:
                for b in self._bns: b.momentum = 1.0          # moving averages unchanged on the 2nd pass
            try:    return self.sub(list(a) if multi else a[0], training=training)
            finally:
                for b, m in zip(self._bns, old): b.momentum = m
        return tf.recompute_grad(run)(*args)

def build_model(hp=HP, gc='none', light=False):
    """gc: 'none' | 'transformer' (the old GC: transformer only) | 'full' (stages 1-3 + transformer)
       light: separable convolutions in stages 2-3 (stage 1 untouched)."""
    k = hp['sa_kernel']; sep = light
    def stage1(w0, w1):
        w0, w1 = spatial_attention(w0, k), spatial_attention(w1, k)
        x1 = cbr(cbr(w0, 64), 64, 2); xa = cbr(w1, 64, 2)
        return layers.Concatenate()([layers.MaxPooling2D(2)(x1), xa])
    def stage2(x, w2):
        w2 = spatial_attention(w2, k)
        x = cbr(cbr(x, 128, 1, sep), 128, 2, sep)
        xb = cbr(cbr(w2, 64, 2), 128, 1, sep)
        return layers.Concatenate()([x, xb])
    def stage3(x, w3):
        w3 = spatial_attention(w3, k)
        x = cbr(cbr(x, 256, 1, sep), 256, 2, sep)
        xc = cbr(cbr(cbr(w3, 64, 2), 256, 1, sep), 256, 1, sep)
        x = layers.Concatenate()([x, xc])
        return cbr(cbr(x, 256, 1, sep), 256, 2, sep)
    def transformer(x):
        x = layers.Reshape((-1, 256))(x)                                   # 3x3 = 9 tokens
        h = layers.LayerNormalization(epsilon=1e-6)(x)
        x = layers.Add()([x, layers.MultiHeadAttention(num_heads=hp['num_heads'], key_dim=hp['key_dim'])(h, h)])
        h = layers.LayerNormalization(epsilon=1e-6)(x)
        h = layers.Dense(hp['ff_dim'], activation='relu')(h); h = layers.Dense(256)(h)
        return layers.Add()([x, h])

    s1 = make_seg('stage1', [(80, 80, 12), (40, 40, 12)], stage1)
    s2 = make_seg('stage2', [(20, 20, 128), (20, 20, 12)], stage2)
    s3 = make_seg('stage3', [(10, 10, 256), (10, 10, 12)], stage3)
    s4 = make_seg('transformer', [(3, 3, 256)], transformer)
    ck_cnn, ck_tr = gc == 'full', gc in ('transformer', 'full')

    inp = Input(shape=(*IMG_SIZE, 3))
    w = [HaarLevel(i, name=f'haar{i}')(inp) for i in (1, 2, 3, 4)]                          # 80x80, 40x40, 20x20, 10x10 (x12 ch)
    x = Remat(s1, ck_cnn, name='remat_stage1')([w[0], w[1]])
    x = Remat(s2, ck_cnn, name='remat_stage2')([x, w[2]])
    x = Remat(s3, ck_cnn, name='remat_stage3')([x, w[3]])
    x = Remat(s4, ck_tr,  name='remat_transformer')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(hp['dropout1'])(layers.BatchNormalization()(layers.Dense(hp['dense1'], activation='relu')(x)))
    x = layers.Dropout(hp['dropout2'])(layers.BatchNormalization()(layers.Dense(hp['dense2'], activation='relu')(x)))
    out = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
    return Model(inp, out)

def n_params(m): return int(m.count_params())


In [ ]:
class Trainer:
    def __init__(self, model, hp=HP, batch=None, micro=None):
        self.m, self.hp = model, hp
        self.batch = batch or hp['batch_size']; self.mb = micro or self.batch
        assert self.batch % self.mb == 0, 'batch must be a multiple of the micro-batch'
        self.n = self.batch // self.mb
        self.vars = model.trainable_variables
        self.opt = tf.keras.optimizers.Adam(hp['lr']); self.opt.build(self.vars)
        self.acc = ([tf.Variable(tf.zeros(v.shape, tf.float32), trainable=False) for v in self.vars]
                    if self.n > 1 else None)
        self.step = tf.function(self._step, reduce_retracing=True)

    def _grads(self, x, y, scale):
        with tf.GradientTape() as t:
            p = tf.cast(self.m(x, training=True), tf.float32)
            loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y[:, None], p))
            scaled = loss * scale                       # MUST be inside the tape, otherwise every gradient is None
        g = t.gradient(scaled, self.vars)
        if any(gi is None for gi in g):                 # never silently train with missing gradients
            raise ValueError(f'{sum(gi is None for gi in g)} of {len(g)} gradients are None - the step would not learn')
        return loss, [tf.cast(gi, tf.float32) for gi in g]

    def _step(self, x, y):
        if self.n == 1:
            total, g = self._grads(x, y, LOSS_SCALE)
        else:
            dep, total = [], 0.
            for j in range(self.n):
                with tf.control_dependencies(dep):                        # micro-batch j starts only after j-1 is accumulated
                    xb = tf.identity(x[j * self.mb:(j + 1) * self.mb]); yb = tf.identity(y[j * self.mb:(j + 1) * self.mb])
                l, gj = self._grads(xb, yb, LOSS_SCALE / self.n)
                dep = [a.assign(gg, read_value=False) if j == 0 else a.assign_add(gg, read_value=False)
                       for a, gg in zip(self.acc, gj)]
                total += l / self.n
            with tf.control_dependencies(dep):
                g = [a.read_value() for a in self.acc]
        finite = tf.reduce_all([tf.reduce_all(tf.math.is_finite(gg)) for gg in g])
        g = [tf.where(finite, gg / LOSS_SCALE, tf.zeros_like(gg)) for gg in g]
        self.opt.apply_gradients(zip(g, self.vars))
        return total

def metrics_from(y_true, y_prob, thr=0.5):
    y_pred = (y_prob > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return dict(accuracy=accuracy_score(y_true, y_pred), precision=precision_score(y_true, y_pred),
                recall=sens, f1=f1_score(y_true, y_pred), specificity=spec,
                gmean=float(np.sqrt(sens * spec)), auc=auc(fpr, tpr))

def predict_all(model, gen):
    gen.reset(); f = tf.function(lambda x: model(x, training=False), reduce_retracing=True); out = []
    for i in range(len(gen)):
        xb, _ = gen[i]; out.append(f(tf.constant(xb)).numpy().reshape(-1))
    return np.concatenate(out)

In [ ]:
import subprocess, sys
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna']); import optuna

HPO_TRIALS, HPO_EPOCHS = (30, 8) if not QUICK_TEST else (2, 1)
FORCE_NEW_SEARCH   = False
COMPARE_WITH_MANUAL = True       # trial 0 = a reference configuration (for a manual-vs-searched comparison)
INPUT_HP_FILE      = None        # optional: best_hyperparameters.json of an earlier session / run
HPO_FILE = f"{OUT_DIR}/best_hyperparameters.json"
HPO_DB   = f"sqlite:///{OUT_DIR}/optuna_study.db"
MANUAL_HP = dict(lr=1e-3, batch_size=64, sa_kernel=7, num_heads=4, ff_dim=128, key_dim=128,
                 dropout1=0.4, dropout2=0.3, dense1=512, dense2=128)       # reference configuration only (not used by the model)
HP_KEYS = list(MANUAL_HP)

def run_search():
    tune_tr, tune_va = train_test_split(df_tune, test_size=0.2, stratify=df_tune['label'], random_state=RANDOM_STATE)
    tune_tr, tune_va = tune_tr.reset_index(drop=True), tune_va.reset_index(drop=True)

    def objective(trial):
        hp = dict(
            lr         = trial.suggest_float('lr', 1e-4, 3e-3, log=True),
            batch_size = trial.suggest_categorical('batch_size', [32, 64]),
            sa_kernel  = trial.suggest_categorical('sa_kernel', [3, 5, 7]),
            num_heads  = trial.suggest_categorical('num_heads', [2, 4, 8]),
            ff_dim     = trial.suggest_categorical('ff_dim', [64, 128, 256]),
            key_dim    = trial.suggest_categorical('key_dim', [32, 64, 128]),
            dropout1   = trial.suggest_float('dropout1', 0.2, 0.5),
            dropout2   = trial.suggest_float('dropout2', 0.1, 0.4),
            dense1     = trial.suggest_categorical('dense1', [256, 512]),
            dense2     = trial.suggest_categorical('dense2', [64, 128]))
        t0 = time.time()
        try:
            tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(RANDOM_STATE)
            trg, vag = make_gens(tune_tr, tune_va, hp['batch_size'])
            model = build_model(hp, gc='none', light=False); T = Trainer(model, hp=hp, micro=hp['batch_size'])
            for ep in range(HPO_EPOCHS):
                for _ in range(len(trg)):
                    xb, yb = next(trg)
                    if len(xb) != hp['batch_size']: continue
                    T.step(tf.constant(xb), tf.constant(yb, tf.float32))
                m = metrics_from(vag.classes, predict_all(model, vag))
                trial.report(m['auc'], ep)                          # the pruner compares the validation AUC after every epoch
                if trial.should_prune(): raise optuna.TrialPruned()
        except tf.errors.ResourceExhaustedError:                    # out of memory for a large configuration
            raise optuna.TrialPruned()
        for k, v in m.items(): trial.set_user_attr(k, float(v))
        trial.set_user_attr('params', int(model.count_params())); trial.set_user_attr('minutes', (time.time() - t0) / 60)
        return m['auc']

    study = optuna.create_study(direction='maximize', study_name='malaria_wavelet_cnn_transformer', storage=HPO_DB, load_if_exists=True,
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=min(5, HPO_TRIALS)),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2))
    if COMPARE_WITH_MANUAL and len(study.trials) == 0: study.enqueue_trial(MANUAL_HP)
    finished = len([t for t in study.trials if t.state in (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)])
    todo = HPO_TRIALS + (1 if COMPARE_WITH_MANUAL else 0) - finished
    if todo > 0: study.optimize(objective, n_trials=todo, gc_after_trial=True)

    trials = study.trials_dataframe(attrs=('number', 'value', 'state', 'params', 'user_attrs'))
    trials.to_csv(f"{OUT_DIR}/hpo_trials.csv", index=False)
    done = trials[trials['state'] == 'COMPLETE'].sort_values('value', ascending=False)
    print('Completed:', len(done), '| pruned:', int((trials['state'] == 'PRUNED').sum()))
    print(done.head(5).to_string(index=False))
    best = {k: type(MANUAL_HP[k])(study.best_params[k]) for k in HP_KEYS}
    if COMPARE_WITH_MANUAL and (trials['number'] == 0).any() and trials.loc[trials['number'] == 0, 'state'].iloc[0] == 'COMPLETE':
        print(f"\nReference configuration (trial 0) validation AUC: {float(trials.loc[trials['number'] == 0, 'value'].iloc[0]):.4f}")
    print(f"Best validation AUC: {study.best_value:.4f} (trial {study.best_trial.number})")
    return best

src = next((p for p in [HPO_FILE, INPUT_HP_FILE] if p and os.path.exists(p) and not FORCE_NEW_SEARCH), None)
if src:
    best = json.load(open(src)); print('Loaded hyperparameters from', src)
else:
    best = run_search()
    json.dump(best, open(HPO_FILE, 'w'), indent=1)
HP.clear(); HP.update(best)
assert set(HP) == set(HP_KEYS), 'incomplete hyperparameters'
print('\nHP =', HP)

# parameter count of the two model variants with the selected hyperparameters
_m = build_model(); print('original pipeline params:', f'{n_params(_m):,}', '| output shape:', _m.output_shape)
_m = build_model(light=True); print('light pipeline params   :', f'{n_params(_m):,}'); del _m


In [ ]:
def gc_correctness_test():
    prev = mixed_precision.global_policy().name
    mixed_precision.set_global_policy('float32'); tf.keras.utils.set_random_seed(0)
    try:
        ref = build_model(gc='none', light=True); ck = build_model(gc='full', light=True)
        ck.set_weights(ref.get_weights())
        for l in ck.layers:
            if isinstance(l, Remat): l.force = True
        x = tf.random.uniform((4, *IMG_SIZE, 3)); y = tf.constant([0., 1., 0., 1.])
        def grads(m, training):
            with tf.GradientTape() as t:
                loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y[:, None], m(x, training=training)))
            return t.gradient(loss, m.trainable_variables)
        ga, gb = grads(ref, False), grads(ck, False)
        rel = max(float(np.max(np.abs(a.numpy() - b.numpy())) / (np.max(np.abs(a.numpy())) + 1e-12))
                  for a, b in zip(ga, gb) if a is not None)
        print(f'max relative gradient difference (GC vs no GC): {rel:.2e} ->', 'PASS' if rel < 1e-3 else 'FAIL')
        # BatchNorm statistics: one training pass, compare the moving means of stage 1
        with tf.GradientTape() as t:
            p = ref(x, training=True); la = tf.reduce_mean(p)
        with tf.GradientTape() as t:
            p = ck(x, training=True); lb = tf.reduce_mean(p)
        bn = lambda m: [b.moving_mean.numpy() for b in m.get_layer('remat_stage1').sub.layers
                        if isinstance(b, layers.BatchNormalization)]
        d = max(float(np.max(np.abs(a - b))) for a, b in zip(bn(ref), bn(ck)))
        print(f'BatchNorm moving-mean difference after one step: {d:.2e} ->', 'PASS' if d < 1e-5 else 'FAIL')
    finally:
        mixed_precision.set_global_policy(prev); tf.keras.backend.clear_session()
gc_correctness_test()

In [ ]:
def learning_check(steps=30):
    ok_all = True
    rng = np.random.default_rng(0)
    X = (rng.random((32, *IMG_SIZE, 3)) * 0.3).astype('float32'); y = np.array([0, 1] * 16, dtype='float32')
    X[y == 1, 40:90, 40:90, 0] = 1.0
    for name, cfg in {'B0': dict(gc='none', light=False, micro=32), 'B4': dict(gc='full', light=True, micro=MICRO_BATCH)}.items():
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(0)
        m = build_model(gc=cfg['gc'], light=cfg['light']); T = Trainer(m, micro=cfg['micro'])
        w0 = [v.numpy().copy() for v in T.vars]
        losses = [float(T.step(tf.constant(X), tf.constant(y))) for _ in range(steps)]
        dw = max(float(np.abs(v.numpy() - a).max()) for v, a in zip(T.vars, w0))
        ok = dw > 0 and np.mean(losses[-5:]) < 0.8 * np.mean(losses[:3])
        ok_all &= ok
        print(f'{name}: loss {np.mean(losses[:3]):.3f} -> {np.mean(losses[-5:]):.3f} | max weight change {dw:.2e} ->', 'PASS' if ok else 'FAIL')
    tf.keras.backend.clear_session()
    return ok_all
assert learning_check(), 'The training step does not learn - do not continue.'

In [ ]:
MEM = lambda: tf.config.experimental.get_memory_info('GPU:0')
MB  = 1024 ** 2

def bench(gc_mode, light, micro, batch=32, warm=3, steps=15):
    tf.keras.backend.clear_session(); gc.collect()
    base = MEM()['current']                                   # whatever is still allocated (should be ~0)
    tf.keras.utils.set_random_seed(0)
    model = build_model(gc=gc_mode, light=light); tr = Trainer(model, batch=batch, micro=micro)
    x = tf.random.uniform((batch, *IMG_SIZE, 3)); y = tf.cast(tf.random.uniform((batch,), 0, 2, tf.int32), tf.float32)
    for _ in range(warm): float(tr.step(x, y))                # trace + cuDNN autotune
    static = MEM()['current'] - base
    tf.config.experimental.reset_memory_stats('GPU:0')
    t0 = time.perf_counter()
    for _ in range(steps): float(tr.step(x, y))
    ms = (time.perf_counter() - t0) / steps * 1000
    peak = MEM()['peak'] - base
    out = dict(params=n_params(model), peak_mb=peak / MB, static_mb=static / MB,
               act_mb=max(peak - static, 0) / MB, ms_step=ms)
    del model, tr, x, y; tf.keras.backend.clear_session(); gc.collect()
    return out

LADDER = {
 'B0 Original, no GC  (baseline)':               dict(gc_mode='none',        light=False, micro=32),
 'B1 + GC on transformer only (old GC)':         dict(gc_mode='transformer', light=False, micro=32),
 'B2 + GC on stages 1-3 + transformer':          dict(gc_mode='full',        light=False, micro=32),
 'B3 + micro-batching (32 = 4 x 8)':             dict(gc_mode='full',        light=False, micro=MICRO_BATCH),
 'B4-noGC  light convs + micro-batch, NO GC':    dict(gc_mode='none',        light=True,  micro=MICRO_BATCH),
 'B4 + light convs  [PROPOSED, with GC]':        dict(gc_mode='full',        light=True,  micro=MICRO_BATCH),
}
bench_res = {}
for name, cfg in LADDER.items():
    print('benchmarking', name, '...'); bench_res[name] = bench(**cfg)

b0 = bench_res['B0 Original, no GC  (baseline)']
rows = []
for name, r in bench_res.items():
    rows.append({'Config': name, 'Params (M)': round(r['params'] / 1e6, 2),
                 'Peak (MB)': round(r['peak_mb'], 1), 'Static (MB)': round(r['static_mb'], 1),
                 'Activations (MB)': round(r['act_mb'], 1), 'ms/step': round(r['ms_step'], 1),
                 'Peak saving %': round(100 * (1 - r['peak_mb'] / b0['peak_mb']), 1),
                 'Activation saving %': round(100 * (1 - r['act_mb'] / max(b0['act_mb'], 1e-9)), 1),
                 'Time overhead %': round(100 * (r['ms_step'] / b0['ms_step'] - 1), 1)})
mem_table = pd.DataFrame(rows)
pd.set_option('display.width', 250, 'display.max_columns', 30, 'display.max_colwidth', 60)
print(mem_table.to_string(index=False))
mem_table.to_csv(f'{OUT_DIR}/memory_ladder.csv', index=False)

prop = bench_res['B4 + light convs  [PROPOSED, with GC]']; nogc = bench_res['B4-noGC  light convs + micro-batch, NO GC']
print(f"\nGC effect inside the final pipeline (B4 vs B4-noGC): peak {100*(1-prop['peak_mb']/nogc['peak_mb']):+.1f} %, "
      f"activations {100*(1-prop['act_mb']/max(nogc['act_mb'],1e-9)):+.1f} %, time {100*(prop['ms_step']/nogc['ms_step']-1):+.1f} %")
for k, lab in [('Peak saving %', 'peak GPU memory'), ('Activation saving %', 'activation memory')]:
    v = float(mem_table.iloc[-1][k]); print(f"{lab:20s}: {v:5.1f} %  ", 'OK (>= target)' if v >= TARGET_SAVING else f'below {TARGET_SAVING:.0f} % -> lower MICRO_BATCH (4) and rerun')

In [ ]:
sweep = []
for mbs in [32, 16, 8, 4]:
    r = bench(gc_mode='full', light=True, micro=mbs)
    sweep.append({'Micro-batch': mbs, 'Peak (MB)': round(r['peak_mb'], 1), 'Activations (MB)': round(r['act_mb'], 1),
                  'ms/step': round(r['ms_step'], 1),
                  'Peak saving % vs B0': round(100 * (1 - r['peak_mb'] / b0['peak_mb']), 1),
                  'Activation saving % vs B0': round(100 * (1 - r['act_mb'] / max(b0['act_mb'], 1e-9)), 1)})
sweep = pd.DataFrame(sweep); print(sweep.to_string(index=False)); sweep.to_csv(f'{OUT_DIR}/microbatch_sweep.csv', index=False)

In [ ]:
def to_tflite(model_fp32, mode):
    path = f'{OUT_DIR}/_tmp_savedmodel'
    if hasattr(model_fp32, 'export'): model_fp32.export(path)
    else: tf.saved_model.save(model_fp32, path)
    c = tf.lite.TFLiteConverter.from_saved_model(path)
    if mode in ('fp16', 'int8'):
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        if mode == 'fp16': c.target_spec.supported_types = [tf.float16]
    return c.convert()

def fp32_copy(model, **cfg):
    prev = mixed_precision.global_policy().name; mixed_precision.set_global_policy('float32')
    try:
        m = build_model(**cfg); m.set_weights(model.get_weights()); return m
    finally: mixed_precision.set_global_policy(prev)

def size_report(name, model, cfg):
    m32 = fp32_copy(model, **cfg); wpath = f'{OUT_DIR}/_w.weights.h5'; m32.save_weights(wpath)
    r = {'Model': name, 'Params (M)': round(n_params(m32) / 1e6, 3), 'Weights fp32 (MB)': round(os.path.getsize(wpath) / MB, 2)}
    blobs = {}
    for mode in ['fp32', 'fp16', 'int8']:
        try: blobs[mode] = to_tflite(m32, mode); r[f'TFLite {mode} (MB)'] = round(len(blobs[mode]) / MB, 2)
        except Exception as e: r[f'TFLite {mode} (MB)'] = np.nan; print(f'  [{name}] TFLite {mode} failed: {type(e).__name__}: {str(e)[:150]}')
    return r, blobs

# models used for size: trained ones if the accuracy section below was run, else freshly built (size does not depend on weights)
size_models = {}
def get_size_model(key, cfg):
    return size_models.get(key) or build_model(**cfg)

CFG_B0 = dict(gc='none', light=False); CFG_B4 = dict(gc='full', light=True)
size_rows, tflite_blobs = [], {}
for key, cfg, label in [('B0', CFG_B0, 'B0 original'), ('B4', CFG_B4, 'B4 proposed (light)')]:
    r, b = size_report(label, get_size_model(key, cfg), cfg); size_rows.append(r); tflite_blobs[key] = b
size_table = pd.DataFrame(size_rows); print(size_table.to_string(index=False))
ob, pb = size_rows[0], size_rows[1]
print('\nModel-size saving of the proposed pipeline vs original float32:')
print(f"  parameters            : {100*(1-pb['Params (M)']/ob['Params (M)']):.1f} %")
print(f"  weights file (fp32)   : {100*(1-pb['Weights fp32 (MB)']/ob['Weights fp32 (MB)']):.1f} %")
for mode in ['fp16', 'int8']:
    if not np.isnan(pb[f'TFLite {mode} (MB)']):
        print(f"  TFLite {mode:4s} vs original TFLite fp32 : {100*(1-pb[f'TFLite {mode} (MB)']/ob['TFLite fp32 (MB)']):.1f} %")
size_table.to_csv(f'{OUT_DIR}/model_size.csv', index=False)

In [ ]:
# ===================== Accuracy over 5 folds: mean ± SD, resumable, paired test =====================
from scipy import stats
RUN_ACCURACY  = True
ACC_FOLDS     = [1, 2, 3, 4, 5]
ACC_EPOCHS    = 15 if not QUICK_TEST else 1
MAX_HOURS     = 10.5      # time budget (hours): the cell stops cleanly and can be rerun to resume
ACC_RESULTS   = f"{OUT_DIR}/accuracy_by_fold.json"
INPUT_RESULTS = None      # optional: path of an accuracy_by_fold.json saved by an earlier session (to resume)
TFLITE_EVAL_N = 1000      # images for the TFLite check (uses the models of the last fold trained; 0 = skip)

ACC_CONFIGS = {'B0 Original, no GC': dict(gc='none', light=False, micro=32),
               'B4 Proposed (GC + micro-batch + light)': dict(gc='full', light=True, micro=MICRO_BATCH)}
PROPOSED = 'B4 Proposed (GC + micro-batch + light)'

BASELINE = 'B0 Original, no GC'

METRICS = ['accuracy', 'precision', 'recall', 'f1', 'specificity', 'gmean', 'auc']
LABEL = dict(accuracy='Accuracy', precision='Precision', recall='Recall', f1='F1', specificity='Specificity', gmean='G-Mean', auc='AUC')

# ---- load previous results (resume) ----
acc_res = {}
for path in [ACC_RESULTS, INPUT_RESULTS]:
    if path and os.path.exists(path): acc_res = json.load(open(path)); print('Loaded', path); break
acc_res.setdefault('_meta', {})
if acc_res['_meta'].get('trainer_version') != 2:          # results from the buggy first version (no learning) must not be reused
    if len(acc_res) > 1: print('Ignoring earlier results produced by an older training step.')
    acc_res = {'_meta': {}}
acc_res['_meta']['trainer_version'] = 2
ph = hashlib.md5("".join(df_cv['filepath']).encode()).hexdigest()
if acc_res['_meta'].get('pool_hash', ph) != ph:
    raise RuntimeError('The image list differs from the earlier session: folds are not comparable.')
acc_res['_meta'].update(pool_hash=ph, epochs=ACC_EPOCHS, hp=HP)

trained, t0, stop = {}, time.time(), False
if RUN_ACCURACY:
    for fold in ACC_FOLDS:
        for name, cfg in ACC_CONFIGS.items():
            if str(fold) in acc_res.get(name, {}): continue
            if (time.time() - t0) / 3600 + 1.3 > MAX_HOURS:
                print('\nTime budget reached - stopping cleanly. Download accuracy_by_fold.json and rerun this cell to resume.')
                stop = True; break
            tr_idx, va_idx = folds[fold - 1]
            tr_df, va_df = df_cv.iloc[tr_idx].reset_index(drop=True), df_cv.iloc[va_idx].reset_index(drop=True)
            tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(RANDOM_STATE + fold)   # one seed per fold
            trg, vag = make_gens(tr_df, va_df, HP['batch_size'])
            model = build_model(gc=cfg['gc'], light=cfg['light']); T = Trainer(model, micro=cfg['micro'])
            print(f'\n=== {name} | fold {fold}/5 | params {n_params(model):,} ===')
            tt = time.time()
            for ep in range(ACC_EPOCHS):
                losses = []
                for _ in range(len(trg)):
                    xb, yb = next(trg)
                    if len(xb) != HP['batch_size']: continue
                    losses.append(float(T.step(tf.constant(xb), tf.constant(yb, tf.float32))))
                print(f'epoch {ep+1:2d}/{ACC_EPOCHS}  loss {np.mean(losses):.4f}  {(time.time()-tt)/60:.1f} min')
            m = metrics_from(vag.classes, predict_all(model, vag))
            acc_res.setdefault(name, {})[str(fold)] = {k: float(v) for k, v in m.items()}
            json.dump(acc_res, open(ACC_RESULTS, 'w'), indent=1)               # saved after every fold
            model.save_weights(f"{OUT_DIR}/w_{name.split()[0]}_fold{fold}.weights.h5")   # reused by the explainability section
            print({k: round(float(v), 4) for k, v in m.items()}, f'| session elapsed {(time.time()-t0)/3600:.2f} h')
            trained[name] = (model, cfg); trained[name + '_va_df'] = va_df
        if stop: break

# ---- baseline for the comparison ----

# ---- tables: per fold, mean ± SD (sample SD, n-1), paired t-test vs baseline ----
names = [n for n in [BASELINE, PROPOSED] if n in acc_res]
per_fold, summary = [], []
for n in names:
    f = acc_res[n]; ks = sorted(f, key=int)
    for k in ks: per_fold.append({'Config': n, 'Fold': int(k), **{LABEL[m]: 100 * f[k][m] for m in METRICS}})
    row = {'Config': n, 'Folds': len(ks)}
    for m in METRICS:
        v = 100 * np.array([f[k][m] for k in ks]); row[LABEL[m]] = f'{v.mean():.2f} ± {v.std(ddof=1):.2f}' if len(v) > 1 else f'{v[0]:.2f}'
    summary.append(row)
per_fold, summary = pd.DataFrame(per_fold), pd.DataFrame(summary)
pd.set_option('display.width', 250, 'display.max_columns', 40)
print('\n=== Mean ± SD over folds (rows with Folds < 5 are INCOMPLETE) ===')
print(summary.to_string(index=False))
for n in names:
    print(f'\n--- {n} ---'); print(per_fold[per_fold['Config'] == n].drop(columns='Config').round(2).to_string(index=False))

if BASELINE in acc_res and PROPOSED in acc_res:
    common = sorted(set(acc_res[BASELINE]) & set(acc_res[PROPOSED]), key=int)
    if len(common) >= 3:
        print(f'\n=== Paired comparison {PROPOSED}  vs  {BASELINE}  ({len(common)} common folds) ===')
        for m in METRICS:
            a = np.array([acc_res[BASELINE][k][m] for k in common]); b = np.array([acc_res[PROPOSED][k][m] for k in common])
            p = stats.ttest_rel(a, b).pvalue if np.any(a != b) else 1.0
            print(f'{LABEL[m]:12s} delta {100*(b-a).mean():+.2f} pp   (paired t-test p = {p:.3f})')
        print('With 5 folds the test has low power: report the deltas and per-fold values, avoid over-claiming.')
per_fold.round(2).to_csv(f'{OUT_DIR}/accuracy_per_fold.csv', index=False); summary.to_csv(f'{OUT_DIR}/accuracy_summary.csv', index=False)

# used by the final summary cell
acc_results = {n: {m: float(np.mean([acc_res[n][k][m] for k in acc_res[n]])) for m in METRICS} for n in names}
for n in names:
    if n in trained: size_models['B0' if n.startswith('B0') else 'B4'] = trained[n][0]

In [ ]:
# TFLite (quantised) accuracy on held-out images of the LAST fold trained in this session - uses the TRAINED models
def tflite_predict(blob, X):
    it = tf.lite.Interpreter(model_content=blob, num_threads=4); it.allocate_tensors()
    i, o = it.get_input_details()[0]['index'], it.get_output_details()[0]['index']; out = []
    for x in X: it.set_tensor(i, x[None].astype(np.float32)); it.invoke(); out.append(float(it.get_tensor(o).reshape(-1)[0]))
    return np.array(out)

tflite_acc = {}
if RUN_ACCURACY and TFLITE_EVAL_N and PROPOSED in trained:
    from PIL import Image
    sub = trained[PROPOSED + '_va_df'].sample(min(TFLITE_EVAL_N, len(trained[PROPOSED + '_va_df'])), random_state=0).reset_index(drop=True)
    X = np.stack([np.asarray(Image.open(p).convert('RGB').resize(IMG_SIZE[::-1], Image.NEAREST), dtype=np.float32) / 255. for p in sub['filepath']])
    y = (sub['label'] == CLASSES[1]).astype(int).values
    todo = [('B4', 'fp32'), ('B4', 'fp16'), ('B4', 'int8')] + ([('B0', 'fp32')] if 'B0 Original, no GC' in trained else [])
    for key, mode in todo:
        name = [n for n in trained if n.startswith(key) and not n.endswith('_va_df')][0]; model, cfg = trained[name]
        blob = to_tflite(fp32_copy(model, gc=cfg['gc'], light=cfg['light']), mode)
        tflite_acc[f'{key} TFLite {mode} ({len(blob)/MB:.2f} MB)'] = metrics_from(y, tflite_predict(blob, X))
    print(pd.DataFrame(tflite_acc).T[['accuracy', 'f1', 'auc']].round(4).to_string())

In [ ]:
from scipy.stats import spearmanr
from PIL import Image

RUN_XAI     = True
XAI_CONFIGS = ['B4 Proposed (GC + micro-batch + light)']      # add 'B0 Original, no GC' to compare with the original model
XAI_FOLDS   = [1]        # fold(s) whose held-out images are explained
N_XAI       = 300 if not QUICK_TEST else 20       # images per fold (half Parasitized, half Uninfected)
XAI_EPOCHS  = 15 if not QUICK_TEST else 1         # only used if no saved weights exist for that fold
GRID, STEPS = 10, 20     # 10x10 patches of 16x16 px; 20 insertion/deletion steps
KEEP_FRAC   = 0.20       # fraction of patches kept for Average Drop / Increase in Confidence
NOISE_STD   = 0.02
HH = IMG_SIZE[0]
XAI_CFGS = {'B0 Original, no GC': dict(gc='none', light=False, micro=32),
            'B4 Proposed (GC + micro-batch + light)': dict(gc='full', light=True, micro=MICRO_BATCH)}

def weights_path(name, fold): return f"{OUT_DIR}/w_{name.split()[0]}_fold{fold}.weights.h5"

def load_xai_images(frame):
    # NEAREST = the Keras generator default, i.e. exactly what the model saw in training
    X = np.stack([np.asarray(Image.open(p).convert('RGB').resize(IMG_SIZE[::-1], Image.NEAREST), dtype=np.float32) / 255.
                  for p in frame['filepath']])
    return X, (frame['label'] == CLASSES[1]).astype(int).values

def minmax(a):
    f = a.reshape(len(a), -1); mn, mx = f.min(1)[:, None, None], f.max(1)[:, None, None]
    return (a - mn) / (mx - mn + 1e-8)

def predict(model, X, bs=128):
    return model.predict(X, batch_size=bs, verbose=0).reshape(-1).astype(np.float32)

def target_prob(p, pred):
    return np.where(pred == 1, p, 1 - p)

# ---------- explanation maps ----------
_avgmax = ChannelAvgMax()
def attention_extractor(model):
    """The 4 spatial-attention modules are single-filter Conv2D layers inside the stage sub-models.
    They are keyed by spatial size (80, 40, 20, 10 = Haar level 1..4); each is applied to the Haar level it was trained on."""
    convs = {}
    for rl in model.layers:
        if isinstance(rl, Remat):
            for l in rl.sub.layers:
                if isinstance(l, layers.Conv2D) and l.filters == 1: convs[int(l.output.shape[1])] = l
    if not convs: return None, 0
    haar = Model(model.input, [model.get_layer(f'haar{i}').output for i in (1, 2, 3, 4)])
    return (haar, convs), len(convs)

def attention_levels(ext, Xb):
    """list of (B, H, W) sigmoid attention maps, level 1 (80x80) first"""
    haar, convs = ext; out = []
    for w in haar(Xb, training=False):
        a = convs[int(w.shape[1])](_avgmax(w))
        out.append(tf.cast(a, tf.float32).numpy()[..., 0])
    return out

def attention_maps(ext, X, bs=64):
    out = []
    for i in range(0, len(X), bs):
        lv = [tf.image.resize(m[..., None], (HH, HH), method='bilinear').numpy()[..., 0] for m in attention_levels(ext, X[i:i + bs])]
        out.append(np.mean([minmax(m) for m in lv], axis=0))      # average of the per-scale maps
    return minmax(np.concatenate(out))

def gradient_maps(model, X, bs=32):
    out = []
    for i in range(0, len(X), bs):
        x = tf.convert_to_tensor(X[i:i + bs])
        with tf.GradientTape() as t:
            t.watch(x); p = tf.cast(model(x, training=False), tf.float32)[:, 0]
        g = tf.reduce_max(tf.abs(t.gradient(p, x)), axis=-1)[..., None]
        out.append(tf.nn.avg_pool2d(g, 9, 1, 'SAME').numpy()[..., 0])
    return minmax(np.concatenate(out))

# ---------- faithfulness ----------
def patch_scores(M):
    s = HH // GRID
    return M.reshape(len(M), GRID, s, GRID, s).mean(axis=(2, 4)).reshape(len(M), -1)

def rank_matrix(scores):
    order = np.argsort(-scores, axis=1); ranks = np.empty_like(order)
    ranks[np.arange(len(scores))[:, None], order] = np.arange(scores.shape[1])[None, :]
    return ranks

def patch_mask(ranks, k):                              # True on the k highest-ranked patches
    m = (ranks < k).reshape(len(ranks), GRID, GRID)
    return np.repeat(np.repeat(m, HH // GRID, axis=1), HH // GRID, axis=2)[..., None]

def trapezoid(v, f):
    v = np.asarray(v); return ((v[1:] + v[:-1]) / 2 * np.diff(f)[:, None]).sum(0)

def insertion_deletion(model, X, pred, ranks):
    n = GRID * GRID; ks = np.linspace(0, n, STEPS + 1).round().astype(int); dele, ins = [], []
    for k in ks:
        m = patch_mask(ranks, k)
        dele.append(target_prob(predict(model, np.where(m, 0., X)), pred))   # remove the top-k patches
        ins.append(target_prob(predict(model, np.where(m, X, 0.)), pred))    # keep only the top-k patches
    f = ks / n
    return trapezoid(dele, f), trapezoid(ins, f)

def drop_and_ic(model, X, pred, p_t, ranks):
    m = patch_mask(ranks, int(KEEP_FRAC * GRID * GRID))
    p_k = target_prob(predict(model, np.where(m, X, 0.)), pred)
    return np.maximum(0, p_t - p_k) / (p_t + 1e-8) * 100, (p_k > p_t).astype(float) * 100

# ---------- localization against a pseudo ground truth ----------
def otsu(v, bins=64):
    h, e = np.histogram(v, bins=bins, range=(0, 1)); h = h.astype(float); c = (e[:-1] + e[1:]) / 2
    w0 = np.cumsum(h); w1 = w0[-1] - w0
    m0 = np.cumsum(h * c) / np.maximum(w0, 1); m1 = (np.sum(h * c) - np.cumsum(h * c)) / np.maximum(w1, 1)
    return e[1:][np.argmax(w0 * w1 * (m0 - m1) ** 2)]        # upper edge of the last 'dark' bin

def cell_and_stain_masks(X):
    cell = X.max(-1) > 0.08                                   # non-black pixels = cell
    stain = np.zeros_like(cell)
    for i in range(len(X)):
        g = X[i, ..., 1][cell[i]]
        if g.size > 50: stain[i] = cell[i] & (X[i, ..., 1] < otsu(g))
    return cell, stain

def localization(M, cell, stain):
    n = len(M); out = {k: np.full(n, np.nan) for k in ['point', 'chance', 'enrich', 'topk', 'cell_focus']}
    for i in range(n):
        m, c, s = M[i], cell[i], stain[i]
        if s.sum() < 20 or c.sum() < 100: continue
        tot = m.sum() + 1e-8; k = int(s.sum())
        out['point'][i]  = float(s.flat[np.argmax(m)]) * 100
        out['chance'][i] = s.mean() * 100
        out['enrich'][i] = (m[s].sum() / tot) / s.mean()
        out['topk'][i]   = s.flat[np.argsort(-m.ravel())[:k]].mean() * 100
        out['cell_focus'][i] = (m[c].sum() / tot) / c.mean()
    return out

# ---------- sparsity and robustness ----------
def gini(M):
    v = np.sort(M.reshape(len(M), -1), axis=1); n = v.shape[1]; i = np.arange(1, n + 1)
    return 2 * (i * v).sum(1) / (n * v.sum(1) + 1e-8) - (n + 1) / n

def norm_entropy(M):
    p = M.reshape(len(M), -1); p = p / (p.sum(1, keepdims=True) + 1e-8)
    return -(p * np.log(p + 1e-12)).sum(1) / np.log(p.shape[1])

def rank_corr(A, B):
    small = lambda a: a.reshape(len(a), 40, HH // 40, 40, HH // 40).mean((2, 4)).reshape(len(a), -1)
    return np.array([spearmanr(x, y)[0] for x, y in zip(small(A), small(B))])

def explain_model(model, ext, X, y, rng):
    p = predict(model, X); pred = (p > 0.5).astype(int); p_t = target_prob(p, pred)
    par = (y == 1) & (pred == 1)                                   # correctly classified Parasitized
    cell, stain = cell_and_stain_masks(X)
    methods = {'Gradient saliency': lambda Z: gradient_maps(model, Z),
               'Random': lambda Z: minmax(rng.random((len(Z), HH, HH)).astype(np.float32))}
    if ext is not None: methods = {'Spatial attention (ours)': lambda Z: attention_maps(ext, Z), **methods}
    res = {}
    for name, fn in methods.items():
        M = fn(X); ranks = rank_matrix(patch_scores(M))
        dl, ins = insertion_deletion(model, X, pred, ranks); ad, ic = drop_and_ic(model, X, pred, p_t, ranks)
        loc = localization(M[par], cell[par], stain[par])
        noisy = fn(np.clip(X + rng.normal(0, NOISE_STD, X.shape).astype(np.float32), 0, 1))
        res[name] = {'Deletion AUC': dl, 'Insertion AUC': ins, 'Insertion - Deletion': ins - dl,
                     'Average Drop %': ad, 'Increase in Conf. %': ic,
                     'Pointing game %': loc['point'], 'Chance pointing %': loc['chance'],
                     'Stain enrichment': loc['enrich'], 'Top-k precision %': loc['topk'], 'Cell focus': loc['cell_focus'],
                     'Gini': gini(M), 'Entropy': norm_entropy(M),
                     'Noise stability': rank_corr(M, noisy),
                     'Flip equivariance': rank_corr(fn(X[:, :, ::-1]), M[:, :, ::-1])}
        res[name]['_maps'] = M
    return res, p, pred

In [ ]:
if RUN_XAI:
    # ============================== EXPLAINABILITY: load (or train once), explain, summarise ==============================
    def get_trained(name, fold):
        cfg = XAI_CFGS[name]; path = weights_path(name, fold)
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(RANDOM_STATE + fold)   # same seed as the accuracy run
        tr_idx, va_idx = folds[fold - 1]
        train_df, val_df = df_cv.iloc[tr_idx].reset_index(drop=True), df_cv.iloc[va_idx].reset_index(drop=True)
        if os.path.exists(path):
            model = build_model(gc='none', light=cfg['light']); model.load_weights(path); print("Loaded weights", path)   # gc is irrelevant for inference
        else:
            print('No saved weights for', name, 'fold', fold, '-> training once with the memory-efficient step')
            model = build_model(gc=cfg['gc'], light=cfg['light']); T = Trainer(model, micro=cfg['micro'])
            trg, _ = make_gens(train_df, val_df, HP['batch_size'])
            for ep in range(XAI_EPOCHS):
                for _ in range(len(trg)):
                    xb, yb = next(trg)
                    if len(xb) == HP['batch_size']: T.step(tf.constant(xb), tf.constant(yb, tf.float32))
                print(f'epoch {ep+1}/{XAI_EPOCHS}')
            model.save_weights(path)
        return model, val_df

    per_image, examples = [], {}
    for fold in XAI_FOLDS:
        for name in XAI_CONFIGS:
            print(f"\n=== Explainability | {name} | fold {fold} ===")
            model, val_df = get_trained(name, fold)
            sub = pd.concat([val_df[val_df['label'] == c].sample(N_XAI // 2, random_state=RANDOM_STATE) for c in CLASSES])
            X, y = load_xai_images(sub); ext, n_att = attention_extractor(model)
            print(f"attention modules found: {n_att} (expected 4) | explained images: {len(X)}")
            res, p, pred = explain_model(model, ext, X, y, np.random.default_rng(RANDOM_STATE))
            print(f"accuracy on the explained sample: {100 * (pred == y).mean():.2f}%")
            if (pred == y).mean() < 0.90:
                print("WARNING: accuracy on the explained sample is below 90% - the model did not train properly "
                      "(or weights were not loaded); explainability numbers would be meaningless. Check before using them.")
            examples[(name, fold)] = (X, y, pred, {k: v['_maps'] for k, v in res.items()})
            for method, d in res.items():
                for metric, vals in d.items():
                    if metric != '_maps':
                        per_image.append(pd.DataFrame({'Config': name, 'Fold': fold, 'Method': method, 'Metric': metric, 'Value': vals}))

    xai_long = pd.concat(per_image, ignore_index=True)
    xai_long.to_csv(f"{OUT_DIR}/xai_per_image.csv", index=False)

    def fmt(v, nd=3, scale=1.0):
        v = np.asarray(v, float) * scale; v = v[~np.isnan(v)]
        return "n/a" if len(v) == 0 else f"{v.mean():.{nd}f} ± {v.std(ddof=1) if len(v) > 1 else 0:.{nd}f}"

    FAITH = [('Deletion AUC', 3), ('Insertion AUC', 3), ('Insertion - Deletion', 3), ('Average Drop %', 1),
             ('Increase in Conf. %', 1), ('Noise stability', 3), ('Flip equivariance', 3)]
    LOC   = [('Pointing game %', 1), ('Chance pointing %', 1), ('Stain enrichment', 2), ('Top-k precision %', 1),
             ('Cell focus', 2), ('Gini', 3), ('Entropy', 3)]
    def table(cols):
        rows = []
        for (cfg, meth), g in xai_long.groupby(['Config', 'Method'], sort=False):
            rows.append({'Config': cfg[:2], 'Method': meth, **{m: fmt(g[g.Metric == m].Value, nd) for m, nd in cols}})
        return pd.DataFrame(rows)
    xai_faith, xai_loc = table(FAITH), table(LOC)
    pd.set_option('display.width', 250, 'display.max_columns', 40)
    print("\n=== Faithfulness and robustness (mean ± SD over images; Deletion/Avg Drop lower is better, others higher) ===")
    print(xai_faith.to_string(index=False))
    print("\n=== Localization against the pseudo ground truth, and sparsity ===")
    print(xai_loc.to_string(index=False))
    xai_faith.to_csv(f"{OUT_DIR}/xai_faithfulness.csv", index=False); xai_loc.to_csv(f"{OUT_DIR}/xai_localization.csv", index=False)

    # ---- paired test: is the attention map more faithful than random / gradient saliency? (per image) ----
    from scipy import stats as _st
    for cfg in XAI_CONFIGS:
        g = xai_long[xai_long.Config == cfg]
        if 'Spatial attention (ours)' not in set(g.Method): continue
        for other in ['Random', 'Gradient saliency']:
            for m in ['Deletion AUC', 'Insertion AUC']:
                a = g[(g.Method == 'Spatial attention (ours)') & (g.Metric == m)].Value.values
                b = g[(g.Method == other) & (g.Metric == m)].Value.values
                if len(a) == len(b) and len(a) > 2:
                    try: pv = f"{_st.wilcoxon(a, b).pvalue:.2e}"
                    except ValueError: pv = "n/a (identical values)"
                    print(f"{cfg[:2]} | attention vs {other:17s} | {m}: mean diff {np.mean(a - b):+.4f}, Wilcoxon p = {pv}")

    # ---- example figure: 4 Parasitized + 4 Uninfected, attention vs gradient saliency ----
    import matplotlib.pyplot as plt
    for (cfg, fold), (X, y, pred, maps) in examples.items():
        idx = list(np.where(y == 1)[0][:4]) + list(np.where(y == 0)[0][:4])
        keys = [k for k in ['Spatial attention (ours)', 'Gradient saliency'] if k in maps]
        fig, ax = plt.subplots(1 + len(keys), len(idx), figsize=(2.2 * len(idx), 2.3 * (1 + len(keys))), squeeze=False)
        for j, i in enumerate(idx):
            ax[0, j].imshow(X[i]); ax[0, j].set_title(f"{CLASSES[y[i]][:4]}→{CLASSES[pred[i]][:4]}", fontsize=8)
            for r, k in enumerate(keys, 1):
                ax[r, j].imshow(X[i]); ax[r, j].imshow(maps[k][i], cmap='jet', alpha=0.45)
                if j == 0: ax[r, j].set_ylabel(k.replace(' (ours)', ''), fontsize=8)
        for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
        plt.tight_layout(); plt.savefig(f"{OUT_DIR}/xai_examples_{cfg.split()[0]}_fold{fold}.png", dpi=200); plt.show()
else:
    print('Explainability section skipped (RUN_XAI = False).')

In [ ]:
# FIGURE: Haar wavelet sub-bands of one colour channel (LL, LH, HL, HH) + spatial attention map of the TRAINED proposed model
SHOW_CLASSES = ['Parasitized']      # e.g. ['Parasitized', 'Uninfected'] -> two rows
SEED_PICK, CHANNEL = 0, 0           # another random image / 0=Red, 1=Green, 2=Blue
ATT_LEVEL    = 1                    # 1 = 80x80 attention module (same size as the level-1 sub-bands); 'mean' = mean of all levels
ATT_NAME, ATT_FOLD = 'B4 Proposed (GC + micro-batch + light)', 1
SAVE_AS      = f"{OUT_DIR}/wavelet_subbands_attention"

def haar1(c):
    a, b = c[:, ::2], c[:, 1::2]; L, H = (a + b) / 2, (a - b) / 2
    return ((L[::2] + L[1::2]) / 2, (L[::2] - L[1::2]) / 2,      # LL, LH (horizontal details)
            (H[::2] + H[1::2]) / 2, (H[::2] - H[1::2]) / 2)      # HL (vertical details), HH (diagonal)

att_model, ext = None, None
if os.path.exists(weights_path(ATT_NAME, ATT_FOLD)):
    tf.keras.backend.clear_session()
    att_model = build_model(gc='none', light=XAI_CFGS[ATT_NAME]['light']); att_model.load_weights(weights_path(ATT_NAME, ATT_FOLD))
    ext, _ = attention_extractor(att_model)
print('Attention source:', weights_path(ATT_NAME, ATT_FOLD) if ext else 'NONE (run the accuracy cell first) -> attention panel left empty')

def attention_map(x):
    lv = attention_levels(ext, x[None])
    if ATT_LEVEL == 'mean':
        nrm = lambda a: (a - a.min()) / (a.max() - a.min() + 1e-8)
        return np.mean([nrm(tf.image.resize(a[0][..., None], lv[0].shape[1:]).numpy()[..., 0]) for a in lv], axis=0)
    return lv[int(ATT_LEVEL) - 1][0]

items = [(c, df_cv[df_cv['label'] == c].sample(1, random_state=SEED_PICK)['filepath'].iloc[0]) for c in SHOW_CLASSES]
R = len(items)
titles = ['Low-Low\n(Approximation)', 'Low-High\n(Horizontal Details)', 'High-Low\n(Vertical Details)',
          'High-High\n(Diagonal Details)', 'Spatial Attention\nMap']
fig, ax = plt.subplots(R, 5, figsize=(11.5, 2.45 * R + 0.55), squeeze=False)
for r, (cls, path) in enumerate(items):
    x = np.asarray(Image.open(path).convert('RGB').resize(IMG_SIZE[::-1], Image.NEAREST), dtype=np.float32) / 255.
    LL, LH, HL, HHb = haar1(x[..., CHANNEL])
    v = np.percentile(np.abs(np.stack([LH, HL, HHb])), 99.5) + 1e-8      # common symmetric range: 0 = mid-gray
    ax[r, 0].imshow(LL, cmap='gray', vmin=0, vmax=1)
    for j, d in enumerate([LH, HL, HHb], 1): ax[r, j].imshow(d, cmap='gray', vmin=-v, vmax=v)
    if ext is not None: ax[r, 4].imshow(attention_map(x), cmap='viridis')
    else: ax[r, 4].text(0.5, 0.5, 'no trained\nmodel', ha='center', va='center', transform=ax[r, 4].transAxes)
    for j in range(5):
        ax[r, j].set_xticks([]); ax[r, j].set_yticks([])
        for s in ax[r, j].spines.values(): s.set_visible(False)
        if r == 0: ax[r, j].set_title(titles[j], fontsize=10)
    if R > 1: ax[r, 0].text(-0.12, 0.5, cls, transform=ax[r, 0].transAxes, rotation=90, ha='center', va='center', fontsize=10)
plt.tight_layout(); plt.savefig(SAVE_AS + ".png", dpi=300, bbox_inches='tight'); plt.savefig(SAVE_AS + ".pdf", bbox_inches='tight')
plt.show(); print("Saved", SAVE_AS + ".png / .pdf")

In [ ]:
import matplotlib.pyplot as plt
last = mem_table.iloc[-1]
size_b4_int8 = pb.get('TFLite int8 (MB)', np.nan); size_b0 = ob['TFLite fp32 (MB)']
summary = {
  'peak_memory_saving_pct':       float(last['Peak saving %']),
  'activation_memory_saving_pct': float(last['Activation saving %']),
  'parameter_saving_pct':         round(100 * (1 - pb['Params (M)'] / ob['Params (M)']), 1),
  'model_size_saving_pct_int8':   (round(100 * (1 - size_b4_int8 / size_b0), 1) if size_b0 and size_b4_int8 == size_b4_int8 else None),
  'step_time_overhead_pct':       float(last['Time overhead %']),
  'GC_effect_in_final_pipeline': {'peak_pct': round(100 * (1 - prop['peak_mb'] / nogc['peak_mb']), 1),
                                  'activation_pct': round(100 * (1 - prop['act_mb'] / max(nogc['act_mb'], 1e-9)), 1)},
  'accuracy_fold': acc_results}
json.dump(summary, open(f'{OUT_DIR}/memory_summary.json', 'w'), indent=1)
print(json.dumps(summary, indent=1))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
short = [n.split(' ')[0] for n in mem_table['Config']]
ax[0].bar(short, mem_table['Peak (MB)'], label='Static (weights, optimiser, buffers)', color='#9aa5b1')
ax[0].bar(short, mem_table['Activations (MB)'], bottom=mem_table['Static (MB)'], label='Activations', color='#2b6cb0')
ax[0].set_ylabel('Peak GPU memory of a training step (MB)'); ax[0].legend(); ax[0].set_title('Peak memory, batch 32')
for i, v in enumerate(mem_table['Peak saving %']): ax[0].text(i, mem_table['Peak (MB)'][i] * 1.01, f'{v:+.0f}%', ha='center', fontsize=9)
ax[1].bar(['original\nfp32', 'light\nfp32', 'light\nfp16', 'light\nint8'],
          [ob['TFLite fp32 (MB)'], pb['TFLite fp32 (MB)'], pb['TFLite fp16 (MB)'], pb['TFLite int8 (MB)']], color=['#9aa5b1', '#2b6cb0', '#2b6cb0', '#2b6cb0'])
ax[1].set_ylabel('Model size on disk (MB)'); ax[1].set_title('Model size (TFLite)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/memory_saving.png', dpi=200); plt.show()